# 08 Spatial Epidemiology — Reference Solutions

Complete solutions for the spatial analysis exercises on the Legionnaires' disease cluster at Songbo Nursing Home.

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

# -- CJK font setup (prevents Chinese labels from showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)
df["died"] = (df["outcome"] == "dead").astype(int)

## Question 1: Spatial Distribution of Case Fatality Rates

In [ ]:
# Compute floor × wing case fatality rates
spatial = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
    died=("died", "sum"),
).reset_index()
spatial["attack_rate"] = (spatial["infected"] / spatial["total"] * 100).round(1)
spatial["cfr"] = (spatial["died"] / spatial["infected"] * 100).round(1)

print("=== Wing attack rates & case fatality rates ===")
print(spatial[["floor", "wing", "total", "infected", "died", "attack_rate", "cfr"]].to_string(index=False))

# CFR heatmap
heatmap_cfr = spatial.pivot(index="floor", columns="wing", values="cfr")

fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(heatmap_cfr, annot=True, fmt=".1f", cmap="Reds",
            cbar_kws={"label": "%"}, ax=ax)
ax.set_title("Case Fatality Rate (%) by Floor × Wing")
ax.set_ylabel("Floor")
plt.tight_layout()
plt.show()

# Interpretation
highest_cfr = spatial.loc[spatial["cfr"].idxmax()]
highest_ar = spatial.loc[spatial["attack_rate"].idxmax()]
print(f"\nHighest CFR: {highest_cfr['floor']}F-{highest_cfr['wing']} ({highest_cfr['cfr']}%)")
print(f"Highest attack rate: {highest_ar['floor']}F-{highest_ar['wing']} ({highest_ar['attack_rate']}%)")
print("\n→ The wing with the highest CFR isn't necessarily the one with the highest attack rate")
print("→ CFR is also affected by resident characteristics (age, comorbidities), not just exposure intensity")

## Question 2: Spatial Distribution of Shower Use

In [ ]:
# Shower-use proportion
shower = df.groupby(["floor", "wing"]).agg(
    total=("case_id", "count"),
    shower_users=("shower_use", "sum"),
    infected=("infected", "sum"),
).reset_index()
shower["shower_pct"] = (shower["shower_users"] / shower["total"] * 100).round(1)
shower["attack_rate"] = (shower["infected"] / shower["total"] * 100).round(1)

print("=== Shower proportion vs. attack rate ===")
print(shower[["floor", "wing", "shower_pct", "attack_rate"]].to_string(index=False))

# Side-by-side heatmaps
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

hm_shower = shower.pivot(index="floor", columns="wing", values="shower_pct")
sns.heatmap(hm_shower, annot=True, fmt=".1f", cmap="Blues",
            cbar_kws={"label": "%"}, ax=axes[0])
axes[0].set_title("Shower-Use Proportion (%)")
axes[0].set_ylabel("Floor")

hm_ar = shower.pivot(index="floor", columns="wing", values="attack_rate")
sns.heatmap(hm_ar, annot=True, fmt=".1f", cmap="YlOrRd",
            cbar_kws={"label": "%"}, ax=axes[1])
axes[1].set_title("Attack Rate (%)")
axes[1].set_ylabel("Floor")

plt.tight_layout()
plt.show()

# Correlation
corr = shower[["shower_pct", "attack_rate"]].corr().iloc[0, 1]
print(f"\nShower proportion vs. attack rate correlation: r = {corr:.3f}")
print("\n→ If the high-low patterns of the two heatmaps look similar, it supports the water-transmission hypothesis")
print("→ But also consider confounders (e.g. functional_status affects shower ability, analyzed in Ch05)")

## Question 3 (Challenge): High-Risk Room List

In [ ]:
# Attack rate for each room
room_stats = df.groupby("room").agg(
    total=("case_id", "count"),
    infected=("infected", "sum"),
).reset_index()
room_stats["attack_rate"] = (room_stats["infected"] / room_stats["total"] * 100).round(1)

# Parse floor and wing
room_stats["floor"] = room_stats["room"].str[0].astype(int)
room_stats["wing"] = room_stats["room"].str[1]

# Filter for >= 75%
high_risk = (
    room_stats[room_stats["attack_rate"] >= 75]
    .sort_values("attack_rate", ascending=False)
    [["room", "total", "infected", "attack_rate", "floor", "wing"]]
)

print(f"=== High-risk room list (attack rate ≥ 75%) ===")
print(f"{len(high_risk)} rooms in total\n")
print(high_risk.to_string(index=False))

# Statistics by wing
print("\n=== Wing distribution of high-risk rooms ===")
wing_counts = high_risk.groupby(["floor", "wing"]).size().reset_index(name="high_risk_rooms")
print(wing_counts.to_string(index=False))

print("\n→ Submit this list to the infection control team to prioritize environmental sampling of these rooms")
print("→ Pay special attention to the wings where high-risk rooms cluster; inspect the showerheads and hot-water piping")

### Interpretation

- **CFR vs. attack rate**: the two aren't necessarily positively correlated. The attack rate reflects exposure risk; the CFR reflects host vulnerability
- **Shower × space**: if the wings with high shower usage are also the ones with high attack rates, the spatial analysis strengthens the water-transmission hypothesis
- **High-risk rooms**: high-risk rooms concentrated in a particular wing suggest that wing's water supply may be the transmission route
- **Recommended action**: culture for Legionella and conduct environmental sampling of the showerheads and hot-water piping in the high-risk wing